# Load notebook dataset

Inspect a small order CSV before replacing notebook-local pandas work with a Dask pipeline step.


In [ ]:
import pandas as pd
from pathlib import Path

orders_path = Path("data/orders.csv")
orders_sample = pd.read_csv(orders_path).head(5)
print(orders_sample)


# Scale feature engineering with Dask

Move row-level revenue calculation and daily aggregation out of notebook memory and into a Dask dataframe plan.


In [ ]:
import dask.dataframe as dd

orders = dd.read_csv("data/orders.csv", blocksize="64MB")
orders["revenue"] = orders["quantity"] * orders["unit_price"]
daily = orders.groupby("order_date").agg({"revenue": "sum", "order_id": "count"}).rename(columns={"order_id": "order_count"}).reset_index()
daily.to_parquet("artifacts/daily_orders.parquet", write_index=False)


# Export analysis summary

Persist summary metrics so AGILAB can render an analysis page without depending on notebook state.


In [ ]:
import json
import dask.dataframe as dd
from pathlib import Path

daily = dd.read_parquet("artifacts/daily_orders.parquet")
summary = daily.describe().compute().to_dict()
Path("artifacts").mkdir(parents=True, exist_ok=True)
Path("artifacts/dask_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
